### __Codigo para encoder de time series__

In [1]:
import numpy as np
import polars as pl
import pandas as pd
from sklearn.base import clone
from copy import deepcopy
import optuna
from scipy.optimize import minimize
import os
import matplotlib.pyplot as plt
import seaborn as sns

import re
from colorama import Fore, Style

from tqdm import tqdm
from IPython.display import clear_output
from concurrent.futures import ThreadPoolExecutor

import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

import lightgbm as lgb
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import *
from sklearn.metrics import *

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler

In [2]:
def process_file(filename, dirname):
    df = pd.read_parquet(os.path.join(dirname, filename, 'part-0.parquet'))
    df.drop('step', axis=1, inplace=True)
    return df.describe().values.reshape(-1), filename.split('=')[1]

def load_time_series(dirname) -> pd.DataFrame:
    ids = os.listdir(dirname)
    
    with ThreadPoolExecutor() as executor:
        results = list(tqdm(executor.map(lambda fname: process_file(fname, dirname), ids), total=len(ids)))
    
    stats, indexes = zip(*results)
    
    df = pd.DataFrame(stats, columns=[f"Stat_{i}" for i in range(len(stats[0]))])
    df['id'] = indexes
    
    return df

In [3]:
train_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_train.parquet")
test_ts = load_time_series("/kaggle/input/child-mind-institute-problematic-internet-use/series_test.parquet")
time_series_cols = train_ts.columns.tolist()
time_series_cols.remove("id")

100%|██████████| 2/2 [00:00<00:00,  9.59it/s]


In [4]:
train = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/train.csv')
test = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/test.csv')
sample = pd.read_csv('/kaggle/input/child-mind-institute-problematic-internet-use/sample_submission.csv')

In [5]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, encoding_dim):
        super(AutoEncoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, encoding_dim*3),
            nn.LeakyReLU(0.2),
            nn.Linear(encoding_dim*3, encoding_dim*2),
            nn.LeakyReLU(0.2),
            nn.Linear(encoding_dim*2, encoding_dim),
            nn.LeakyReLU(0.2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, input_dim*2),
            nn.LeakyReLU(0.2),
            nn.Linear(input_dim*2, input_dim*3),
            nn.LeakyReLU(0.2),
            nn.Linear(input_dim*3, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [6]:
def perform_autoencoder(df, encoding_dim=50, epochs=50, batch_size=32):
    scaler = StandardScaler()
    df_scaled = scaler.fit_transform(df)

    data_tensor = torch.FloatTensor(df_scaled)

    input_dim = data_tensor.shape[1]
    autoencoder = AutoEncoder(input_dim, encoding_dim)

    criterion = F.smooth_l1_loss
    optimizer = optim.Adam(autoencoder.parameters())

    for epoch in range(epochs):
        for i in range(0, len(data_tensor), batch_size):
            batch = data_tensor[i : i + batch_size]
            optimizer.zero_grad()
            reconstructed = autoencoder(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}]')

    with torch.no_grad():
        encoded_data = autoencoder.encoder(data_tensor).numpy()

    df_encoded = pd.DataFrame(encoded_data, columns=[f'Enc_{i + 1}' for i in range(encoded_data.shape[1])])

    return df_encoded

In [7]:
df_train = train_ts.drop('id', axis=1)
df_test = test_ts.drop('id', axis=1)

train_ts_encoded = perform_autoencoder(df_train, encoding_dim=60, epochs=100, batch_size=32)
test_ts_encoded = perform_autoencoder(df_test, encoding_dim=60, epochs=100, batch_size=32)

time_series_cols = train_ts_encoded.columns.tolist()

train_ts_encoded["id"] = train_ts["id"]
test_ts_encoded["id"] = test_ts["id"]
train_ts = train_ts_encoded
test_ts = test_ts_encoded

Epoch [10/100], Loss: 0.4819]
Epoch [20/100], Loss: 0.4656]
Epoch [30/100], Loss: 0.4583]
Epoch [40/100], Loss: 0.4505]
Epoch [50/100], Loss: 0.4503]
Epoch [60/100], Loss: 0.4474]
Epoch [70/100], Loss: 0.4462]
Epoch [80/100], Loss: 0.4418]
Epoch [90/100], Loss: 0.4417]
Epoch [100/100], Loss: 0.4425]
Epoch [10/100], Loss: 0.4292]
Epoch [20/100], Loss: 0.2135]
Epoch [30/100], Loss: 0.2135]
Epoch [40/100], Loss: 0.2135]
Epoch [50/100], Loss: 0.2135]
Epoch [60/100], Loss: 0.2135]
Epoch [70/100], Loss: 0.2135]
Epoch [80/100], Loss: 0.2135]
Epoch [90/100], Loss: 0.2135]
Epoch [100/100], Loss: 0.2135]


In [8]:
train_ts

,Enc_1,Enc_2,Enc_3,Enc_4,Enc_5,Enc_6,Enc_7,Enc_8,Enc_9,Enc_10,Enc_11,Enc_12,Enc_13,Enc_14,Enc_15,Enc_16,Enc_17,Enc_18,Enc_19,Enc_20,Enc_21,Enc_22,Enc_23,Enc_24,Enc_25,Enc_26,Enc_27,Enc_28,Enc_29,Enc_30,Enc_31,Enc_32,Enc_33,Enc_34,Enc_35,Enc_36,Enc_37,Enc_38,Enc_39,Enc_40,Enc_41,Enc_42,Enc_43,Enc_44,Enc_45,Enc_46,Enc_47,Enc_48,Enc_49,Enc_50,Enc_51,Enc_52,Enc_53,Enc_54,Enc_55,Enc_56,Enc_57,Enc_58,Enc_59,Enc_60,id
0,3.959449,-0.226655,0.278408,0.604870,-0.552456,3.093329,0.867858,1.888699,-0.357958,-0.567262,-0.061805,-1.358103,1.106034,2.128851,3.908282,-0.005716,-0.090693,4.605579,0.438190,-0.955551,-0.196209,3.556526,-0.166909,-0.366779,-0.165272,-0.015795,3.901603,-0.557541,-0.071282,2.183322,0.732970,2.904383,-0.164448,-0.380411,2.126330,0.111249,-0.015422,0.740242,4.005163,0.435063,0.974850,-0.793684,-0.694790,3.023711,-0.877695,-0.437573,0.701363,0.945439,-0.476549,1.832272,-0.490612,0.489699,4.729598,0.409489,-0.720045,5.589021,-0.421643,1.485616,2.455581,0.587143,0745c390
1,3.760564,-0.118855,0.443124,3.362604,-0.851709,0.204559,1.345113,-0.321490,1.677160,-0.844325,3.352568,0.021656,2.715545,2.473856,-0.554304,2.873112,0.251287,-0.915381,-0.232700,-0.460176,-0.037735,3.889574,0.613659,0.501376,-0.304981,1.021362,-0.462149,2.053576,0.958834,-0.151595,-0.501826,-0.529232,-0.073216,3.034712,1.649980,3.043911,2.834934,2.436265,-0.076703,6.289393,-0.575266,2.512134,0.258976,-0.216522,0.077040,-0.510448,0.044365,2.220837,-0.068502,-0.150497,-0.828336,2.828743,-0.109288,2.281693,3.235226,3.277000,1.966034,-0.130959,-0.293727,-0.132188,eaab7a96
2,0.530424,-0.220165,3.820170,1.975042,1.966758,3.058760,-0.265048,5.501397,-0.054467,0.058180,-0.161369,-0.383020,-0.130791,2.472924,3.161281,-0.399445,2.091847,2.439519,-1.040061,-0.537568,5.025787,1.728245,-0.173972,-0.299935,3.634292,0.690329,-0.464284,5.647570,-1.124421,5.288262,-0.045257,0.010004,-0.037886,5.456063,3.318202,0.559582,-0.426679,3.876721,-0.204267,1.390106,6.623382,-0.714622,-0.643652,2.418228,-1.134912,0.842188,3.318053,-0.089734,3.064026,-0.044366,1.158029,-0.536633,2.157230,0.325072,-0.411797,9.189453,-1.376390,2.695822,-0.808620,0.298231,8ec2cc63
3,0.347043,3.746310,1.414820,-0.134583,-0.308444,3.550384,-0.151196,1.693937,-0.766598,-0.997719,0.045140,-1.677476,-0.691304,12.250333,7.184070,-0.559346,-0.390850,5.446438,-0.505498,0.139183,-0.457391,5.981662,-0.471112,-0.317466,4.014472,1.075392,5.507353,-0.787548,1.435486,2.259878,0.677364,5.652678,0.302740,2.724470,1.223165,0.037968,8.178198,0.714983,5.524651,-0.523603,-0.048380,-1.153840,-0.171679,3.338070,-0.807573,-0.666469,0.813557,-0.141816,3.833503,1.106250,-0.509415,-1.050836,0.831090,-0.060827,-0.575351,2.672610,-0.786718,1.495732,3.302739,4.754278,b2987a65
4,-0.112635,1.783958,5.449903,-0.019898,-0.092836,6.691418,-0.238046,1.385117,-1.043341,0.199816,1.250592,-1.133402,-0.533337,4.881582,2.661589,1.924252,3.508824,6.479623,1.145959,-0.234997,-0.713489,-0.016009,-0.159544,-0.306009,3.361357,-0.322920,5.892976,0.393034,3.491300,0.951928,0.737568,0.705207,-0.099144,-0.410647,0.629757,-0.440722,-0.372610,1.006234,3.024198,-0.556702,2.516646,-1.113494,-0.661016,3.947154,-0.584440,2.261943,6.610763,3.041170,2.056331,-0.279780,-0.185116,-0.941239,1.431258,1.029921,-0.431683,0.700870,-0.784883,0.251246,0.161394,0.259895,7b8842c3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
991,-0.565831,-0.464016,3.094149,-0.267461,-0.372611,1.400561,3.508291,-0.548961,-0.703149,2.590806,1.649559,1.260795,1.892541,2.222411,1.076715,2.223403,2.614870,0.487357,-0.002980,2.403648,2.688730,-0.122946,0.483740,1.544449,-0.541251,-0.597359,-0.556867,0.370090,-0.139755,-0.001526,-0.021572,-0.889065,-0.010515,0.184282,-0.443993,1.034331,-0.194302,-0.202711,-0.215282,2.680057,-0.368059,-0.362924,-0.029158,-0.440775,-0.301078,4.324595,4.17274

#### Unir en un unico df

In [9]:
train = pd.merge(train, train_ts, how="left", on='id')
test = pd.merge(test, test_ts, how="left", on='id')

In [10]:
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

In [11]:
train = train.dropna(subset='sii')

Selecciona las columnas que estan en test y train para entrenar 

In [12]:
def remove_outliers(df):
    
    df = df.drop(df[df['Physical-BMI'] <= 0].index)
    df = df.drop(df[df['Physical-Diastolic_BP'] <= 0].index)
    df = df.drop(df[df['Physical-Systolic_BP'] <= 0].index)
    df = df.drop(df[df['Physical-Diastolic_BP'] > 160].index)

    children = df[df['Basic_Demos-Age'] <= 12]
    df = df.drop(children[children['FGC-FGC_CU'] > 80].index)
    df = df.drop(children[children['FGC-FGC_GSND'] > 80].index)

    df = df.drop(df[df['BIA-BIA_BMI'] <= 0].index)
    df = df.drop(df[df['BIA-BIA_BMC'] > 1000].index)
    df = df.drop(df[df['BIA-BIA_BMR'] > 40000].index)
    df = df.drop(df[df['BIA-BIA_DEE'] > 60000].index)
    df = df.drop(df[df['BIA-BIA_ECW'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_FFM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_ICW'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_LDM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_LST'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_SMM'] > 2000].index)
    df = df.drop(df[df['BIA-BIA_TBW'] > 2000].index)
    
    return df

In [13]:
featuresCols = ['Basic_Demos-Enroll_Season', 'Basic_Demos-Age', 'Basic_Demos-Sex',
                'CGAS-Season', 'CGAS-CGAS_Score', 'Physical-Season', 'Physical-BMI',
                'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference',
                'Physical-Diastolic_BP', 'Physical-HeartRate', 'Physical-Systolic_BP',
                'Fitness_Endurance-Season', 'Fitness_Endurance-Max_Stage',
                'Fitness_Endurance-Time_Mins', 'Fitness_Endurance-Time_Sec',
                'FGC-Season', 'FGC-FGC_CU', 'FGC-FGC_CU_Zone', 'FGC-FGC_GSND',
                'FGC-FGC_GSND_Zone', 'FGC-FGC_GSD', 'FGC-FGC_GSD_Zone', 'FGC-FGC_PU',
                'FGC-FGC_PU_Zone', 'FGC-FGC_SRL', 'FGC-FGC_SRL_Zone', 'FGC-FGC_SRR',
                'FGC-FGC_SRR_Zone', 'FGC-FGC_TL', 'FGC-FGC_TL_Zone', 'BIA-Season',
                'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMI',
                'BIA-BIA_BMR', 'BIA-BIA_DEE', 'BIA-BIA_ECW', 'BIA-BIA_FFM',
                'BIA-BIA_FFMI', 'BIA-BIA_FMI', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num',
                'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM',
                'BIA-BIA_TBW', 'PAQ_A-Season', 'PAQ_A-PAQ_A_Total', 'PAQ_C-Season',
                'PAQ_C-PAQ_C_Total', 'SDS-Season', 'SDS-SDS_Total_Raw',
                'SDS-SDS_Total_T', 'PreInt_EduHx-Season',
                'PreInt_EduHx-computerinternet_hoursday', 'sii']

featuresCols += time_series_cols

train = train[featuresCols]

In [14]:
def feature_engineering(df):
    season_cols = [col for col in df.columns if 'Season' in col]
    df = df.drop(season_cols, axis=1) 
    df['BMI_Age'] = df['Physical-BMI'] * df['Basic_Demos-Age']
    df['Internet_Hours_Age'] = df['PreInt_EduHx-computerinternet_hoursday'] * df['Basic_Demos-Age']
    df['BMI_Internet_Hours'] = df['Physical-BMI'] * df['PreInt_EduHx-computerinternet_hoursday']
    df['BFP_BMI'] = df['BIA-BIA_Fat'] / df['BIA-BIA_BMI']
    df['FFMI_BFP'] = df['BIA-BIA_FFMI'] / df['BIA-BIA_Fat']
    df['FMI_BFP'] = df['BIA-BIA_FMI'] / df['BIA-BIA_Fat']
    df['LST_TBW'] = df['BIA-BIA_LST'] / df['BIA-BIA_TBW']
    df['BFP_BMR'] = df['BIA-BIA_Fat'] * df['BIA-BIA_BMR']
    df['BFP_DEE'] = df['BIA-BIA_Fat'] * df['BIA-BIA_DEE']
    df['BMR_Weight'] = df['BIA-BIA_BMR'] / df['Physical-Weight']
    df['DEE_Weight'] = df['BIA-BIA_DEE'] / df['Physical-Weight']
    df['SMM_Height'] = df['BIA-BIA_SMM'] / df['Physical-Height']
    df['Muscle_to_Fat'] = df['BIA-BIA_SMM'] / df['BIA-BIA_FMI']
    df['Hydration_Status'] = df['BIA-BIA_TBW'] / df['Physical-Weight']
    df['ICW_TBW'] = df['BIA-BIA_ICW'] / df['BIA-BIA_TBW']
    
    return df